# 🏦 Loan Approval Prediction
### Machine Learning Project

---

**Objective:** Predict whether a loan application will be approved or rejected based on applicant details using Machine Learning.

**Dataset:** Home Loan Approval Dataset (Kaggle)

**Models Used:**
- Logistic Regression
- Decision Tree Classifier
- Random Forest Classifier

**Workflow:**
1. Data Collection
2. Exploratory Data Analysis (EDA)
3. Data Preprocessing
4. Model Building
5. Model Evaluation & Comparison
6. Conclusion

## 📦 Step 1: Install Dependencies & Import Libraries

In [ ]:
# Install kagglehub to download dataset
!pip install kagglehub -q

In [ ]:
# ─── Core Libraries ───────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ─── Visualization ────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Set2')

# ─── Preprocessing ────────────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing    import LabelEncoder, StandardScaler
from sklearn.impute            import SimpleImputer

# ─── Models ───────────────────────────────────────────────────
from sklearn.linear_model    import LogisticRegression
from sklearn.tree            import DecisionTreeClassifier, plot_tree
from sklearn.ensemble        import RandomForestClassifier

# ─── Evaluation ───────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve,
    ConfusionMatrixDisplay
)

print('✅ All libraries imported successfully!')

## 📥 Step 2: Load Dataset

In [ ]:

import os

# Download dataset from Kaggle
path = kagglehub.dataset_download('rishikeshkonapure/home-loan-approval')
print('Path to dataset files:', path)

# List files available
for f in os.listdir(path):
    print(' -', f)

In [ ]:
# Load the CSV file
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
df = pd.read_csv(os.path.join(path, csv_files[0]))

print(f'Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns')
df.head(10)

## 🔍 Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# ── Basic Info ────────────────────────────────────────────────
print('='*55)
print('DATASET INFORMATION')
print('='*55)
df.info()

In [ ]:
# ── Statistical Summary ───────────────────────────────────────
print('STATISTICAL SUMMARY')
df.describe(include='all').T

In [ ]:
# ── Missing Values ────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print('MISSING VALUES SUMMARY')
print(missing_df if not missing_df.empty else 'No missing values found!')

# Plot missing values
if not missing_df.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    missing_df['Missing %'].plot(kind='barh', ax=ax, color='#e76f51')
    ax.set_xlabel('Missing %')
    ax.set_title('Missing Values per Column')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Target Variable Distribution ──────────────────────────────
# Detect target column automatically
target_col = None
for c in df.columns:
    if 'status' in c.lower() or 'loan_s' in c.lower() or c.lower() == 'loan_status':
        target_col = c
        break
if target_col is None:
    target_col = df.columns[-1]   # fallback: last column

print(f'Target column detected: "{target_col}"')
print(df[target_col].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

vc = df[target_col].value_counts()
vc.plot(kind='bar', ax=axes[0], color=['#2a9d8f','#e76f51'], edgecolor='white', width=0.5)
axes[0].set_title('Loan Approval Count')
axes[0].set_xlabel('Loan Status')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

vc.plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
        colors=['#2a9d8f','#e76f51'], startangle=90,
        wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_ylabel('')
axes[1].set_title('Loan Approval Distribution')

plt.suptitle('Target Variable: Loan Status', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Categorical Features vs Loan Status ───────────────────────
cat_cols = df.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c != target_col]

n = len(cat_cols)
cols_per_row = 3
rows = (n + cols_per_row - 1) // cols_per_row

fig, axes = plt.subplots(rows, cols_per_row, figsize=(15, 4 * rows))
axes = axes.flatten() if n > 1 else [axes]

for i, col in enumerate(cat_cols):
    ct = pd.crosstab(df[col], df[target_col], normalize='index') * 100
    ct.plot(kind='bar', ax=axes[i], edgecolor='white',
            color=['#2a9d8f','#e76f51'], width=0.6)
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Percentage (%)')
    axes[i].tick_params(axis='x', rotation=30)
    axes[i].legend(title=target_col, fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Categorical Features vs Loan Status', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Numerical Features Distribution ───────────────────────────
num_cols = df.select_dtypes(include=['int64','float64']).columns.tolist()

n = len(num_cols)
cols_per_row = 3
rows = (n + cols_per_row - 1) // cols_per_row

fig, axes = plt.subplots(rows, cols_per_row, figsize=(15, 4 * rows))
axes = axes.flatten() if n > 1 else [axes]

for i, col in enumerate(num_cols):
    for label, color in zip(df[target_col].unique(), ['#2a9d8f','#e76f51']):
        subset = df[df[target_col] == label][col].dropna()
        axes[i].hist(subset, bins=25, alpha=0.6, color=color, label=str(label), edgecolor='white')
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')
    axes[i].legend(fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numerical Features Distribution by Loan Status', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation Heatmap ────────────────────────────────────────
num_df = df[num_cols].copy()

fig, ax = plt.subplots(figsize=(10, 7))
mask = np.triu(np.ones_like(num_df.corr(), dtype=bool))
sns.heatmap(
    num_df.corr(), mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, ax=ax,
    linewidths=0.5, linecolor='white',
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Correlation Heatmap (Numerical Features)', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

## ⚙️ Step 4: Data Preprocessing

In [ ]:
# ── Drop Loan_ID column if present ────────────────────────────
df_clean = df.copy()
id_cols = [c for c in df_clean.columns if 'id' in c.lower()]
if id_cols:
    df_clean.drop(columns=id_cols, inplace=True)
    print(f'Dropped ID columns: {id_cols}')

print(f'Shape after dropping ID: {df_clean.shape}')

In [ ]:
# ── Handle Missing Values ──────────────────────────────────────
# Categorical → fill with mode
for col in df_clean.select_dtypes(include='object').columns:
    if df_clean[col].isnull().sum() > 0:
        mode_val = df_clean[col].mode()[0]
        df_clean[col].fillna(mode_val, inplace=True)
        print(f'  [Categorical] {col}: filled with mode "{mode_val}"')

# Numerical → fill with median
for col in df_clean.select_dtypes(include=['int64','float64']).columns:
    if df_clean[col].isnull().sum() > 0:
        med_val = df_clean[col].median()
        df_clean[col].fillna(med_val, inplace=True)
        print(f'  [Numerical]   {col}: filled with median {med_val}')

print(f'\nMissing values remaining: {df_clean.isnull().sum().sum()}')

In [ ]:
# ── Encode Categorical Features ───────────────────────────────
le = LabelEncoder()
for col in df_clean.select_dtypes(include='object').columns:
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))
    print(f'  Encoded: {col}')

print(f'\nShape after encoding: {df_clean.shape}')
df_clean.head()

In [ ]:
# ── Feature / Target Split ─────────────────────────────────────
X = df_clean.drop(columns=[target_col])
y = df_clean[target_col]

print(f'Features (X): {X.shape}')
print(f'Target  (y): {y.shape}')
print(f'Target classes: {y.unique()}')

In [ ]:
# ── Train / Test Split (80 / 20) ───────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training set  : {X_train.shape}')
print(f'Testing  set  : {X_test.shape}')

In [ ]:
# ── Feature Scaling (for Logistic Regression) ─────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('Feature scaling applied (StandardScaler).')

## 🤖 Step 5: Model Building & Training

### 5.1 Logistic Regression

In [ ]:
# ── Logistic Regression ────────────────────────────────────────
# A linear model that estimates the probability of a binary outcome.
# Uses the sigmoid function to output values between 0 and 1.

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_sc, y_train)          # Scaled features
lr_pred  = lr_model.predict(X_test_sc)
lr_prob  = lr_model.predict_proba(X_test_sc)[:, 1]

lr_acc   = accuracy_score(y_test, lr_pred)
lr_auc   = roc_auc_score(y_test, lr_prob)

print(f'Logistic Regression — Accuracy: {lr_acc:.4f} | AUC: {lr_auc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, lr_pred))

### 5.2 Decision Tree

In [ ]:
# ── Decision Tree ──────────────────────────────────────────────
# Tree-based model that splits data on feature thresholds.
# max_depth controls overfitting.

dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)             # No scaling needed
dt_pred  = dt_model.predict(X_test)
dt_prob  = dt_model.predict_proba(X_test)[:, 1]

dt_acc   = accuracy_score(y_test, dt_pred)
dt_auc   = roc_auc_score(y_test, dt_prob)

print(f'Decision Tree  — Accuracy: {dt_acc:.4f} | AUC: {dt_auc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, dt_pred))

In [ ]:
# ── Visualize Decision Tree ────────────────────────────────────
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(
    dt_model, feature_names=X.columns.tolist(),
    class_names=[str(c) for c in dt_model.classes_],
    filled=True, rounded=True, max_depth=3, ax=ax,
    fontsize=9
)
ax.set_title('Decision Tree (depth ≤ 3 shown)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.3 Random Forest

In [ ]:
# ── Random Forest ──────────────────────────────────────────────
# Ensemble of Decision Trees trained on bootstrapped subsets.
# Reduces variance through averaging — typically the strongest model here.

rf_model = RandomForestClassifier(n_estimators=100, max_depth=6,
                                   random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_pred  = rf_model.predict(X_test)
rf_prob  = rf_model.predict_proba(X_test)[:, 1]

rf_acc   = accuracy_score(y_test, rf_pred)
rf_auc   = roc_auc_score(y_test, rf_prob)

print(f'Random Forest  — Accuracy: {rf_acc:.4f} | AUC: {rf_auc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, rf_pred))

## 📊 Step 6: Model Evaluation & Comparison

In [ ]:
# ── Confusion Matrices ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

models_info = [
    ('Logistic Regression', lr_pred, '#2a9d8f'),
    ('Decision Tree',       dt_pred, '#e9c46a'),
    ('Random Forest',       rf_pred, '#e76f51'),
]

for ax, (name, pred, color) in zip(axes, models_info):
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=11, fontweight='bold')

plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── ROC Curves ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

roc_models = [
    ('Logistic Regression', lr_prob, '#2a9d8f', '-'),
    ('Decision Tree',       dt_prob, '#e9c46a', '--'),
    ('Random Forest',       rf_prob, '#e76f51', '-.'),
]

for name, prob, color, ls in roc_models:
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc_score   = roc_auc_score(y_test, prob)
    ax.plot(fpr, tpr, linestyle=ls, color=color, lw=2.5,
            label=f'{name}  (AUC = {auc_score:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve Comparison', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.show()

In [ ]:
# ── Cross-Validation Scores ────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {}
for name, model, X_data in [
    ('Logistic Regression', lr_model, X_train_sc),
    ('Decision Tree',       dt_model, X_train),
    ('Random Forest',       rf_model, X_train),
]:
    scores = cross_val_score(model, X_data, y_train, cv=cv, scoring='accuracy')
    cv_results[name] = scores
    print(f'{name:<22} CV Accuracy: {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
# ── Feature Importance (Random Forest) ────────────────────────
feat_imp = pd.Series(rf_model.feature_importances_, index=X.columns)
feat_imp = feat_imp.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#e76f51' if v == feat_imp.max() else '#2a9d8f' for v in feat_imp]
feat_imp.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.set_title('Feature Importance — Random Forest', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')

patch1 = mpatches.Patch(color='#e76f51', label='Most important')
patch2 = mpatches.Patch(color='#2a9d8f', label='Other features')
ax.legend(handles=[patch1, patch2], fontsize=9)

plt.tight_layout()
plt.show()

print('\nTop 5 Features:')
print(feat_imp.sort_values(ascending=False).head())

In [ ]:
# ── Model Comparison Summary Table ────────────────────────────
from sklearn.metrics import precision_score, recall_score, f1_score

summary = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest'],
    'Accuracy' : [lr_acc,  dt_acc,  rf_acc],
    'Precision': [
        precision_score(y_test, lr_pred, average='weighted', zero_division=0),
        precision_score(y_test, dt_pred, average='weighted', zero_division=0),
        precision_score(y_test, rf_pred, average='weighted', zero_division=0),
    ],
    'Recall'   : [
        recall_score(y_test, lr_pred, average='weighted'),
        recall_score(y_test, dt_pred, average='weighted'),
        recall_score(y_test, rf_pred, average='weighted'),
    ],
    'F1-Score' : [
        f1_score(y_test, lr_pred, average='weighted'),
        f1_score(y_test, dt_pred, average='weighted'),
        f1_score(y_test, rf_pred, average='weighted'),
    ],
    'AUC-ROC'  : [lr_auc, dt_auc, rf_auc],
    'CV Mean'  : [cv_results['Logistic Regression'].mean(),
                  cv_results['Decision Tree'].mean(),
                  cv_results['Random Forest'].mean()],
}).set_index('Model').round(4)

print('='*70)
print('MODEL COMPARISON SUMMARY')
print('='*70)
summary

In [ ]:
# ── Bar Chart: Accuracy Comparison ────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
model_names = summary.index.tolist()
accuracies  = summary['Accuracy'].values
bar_colors  = ['#2a9d8f', '#e9c46a', '#e76f51']

bars = ax.bar(model_names, accuracies, color=bar_colors, width=0.45, edgecolor='white')

for bar, val in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylim(0, 1.12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Model Accuracy Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 🔮 Step 7: Predict on New Data

In [ ]:
# ── Predict a single applicant using the best model ─────────────
# Identify best model
best_model_name = summary['Accuracy'].idxmax()
print(f'Best Model: {best_model_name} (Accuracy = {summary.loc[best_model_name, "Accuracy"]:.4f})')

# Use a sample from test set as a demo
sample = X_test.iloc[[0]]
sample_scaled = scaler.transform(sample)  # scale for LR

if best_model_name == 'Logistic Regression':
    prediction = lr_model.predict(sample_scaled)
    probability = lr_model.predict_proba(sample_scaled)[0][1]
elif best_model_name == 'Decision Tree':
    prediction = dt_model.predict(sample)
    probability = dt_model.predict_proba(sample)[0][1]
else:
    prediction = rf_model.predict(sample)
    probability = rf_model.predict_proba(sample)[0][1]

print(f'\nPrediction : {"✅ Approved" if prediction[0] == 1 else "❌ Rejected"}')
print(f'Probability: {probability:.2%}')

## 📝 Step 8: Conclusion

In [ ]:
best = summary['Accuracy'].idxmax()
best_acc = summary.loc[best, 'Accuracy']
best_auc = summary.loc[best, 'AUC-ROC']

print('='*60)
print('       LOAN APPROVAL PREDICTION — CONCLUSION')
print('='*60)
print()
print('This project built and compared three ML classifiers')
print('to predict whether a home loan will be approved.')
print()
print('Key Steps Performed:')
print('  1. Loaded dataset via KaggleHub')
print('  2. Performed EDA (distributions, correlations, missing values)')
print('  3. Handled missing values (mode/median imputation)')
print('  4. Encoded categorical variables (LabelEncoder)')
print('  5. Scaled features (StandardScaler for LR)')
print('  6. Trained 3 models: LR, Decision Tree, Random Forest')
print('  7. Evaluated using Accuracy, Precision, Recall, F1, AUC')
print('  8. 5-Fold Cross-Validation for generalization check')
print()
print(f'Best Performing Model : {best}')
print(f'  Test Accuracy        : {best_acc:.4f}')
print(f'  AUC-ROC              : {best_auc:.4f}')
print()
print('Top Loan Approval Factors (from Random Forest):')
top_features = pd.Series(rf_model.feature_importances_, index=X.columns)\
               .sort_values(ascending=False).head(3)
for feat, imp in top_features.items():
    print(f'  - {feat}: {imp:.4f}')
print()
print('Future Improvements:')
print('  - Hyperparameter tuning with GridSearchCV')
print('  - Try XGBoost / SVM for higher accuracy')
print('  - Handle class imbalance with SMOTE')
print('  - Deploy as a web app using Streamlit/Flask')
print('='*60)